In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import io
import base64
import threading
import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from flask import Flask, request, jsonify, render_template_string
from sklearn.ensemble import RandomForestRegressor
from werkzeug.serving import make_server

# -----------------------------
# Settings
# -----------------------------
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

file_path = r"C:\Users\Acer\Downloads\Clothing Sales Data.csv"
FLASK_PORT = 5001

app = Flask(__name__)

# -----------------------------
# Silence Flask / Werkzeug logs
# -----------------------------
werkzeug_log = logging.getLogger("werkzeug")
werkzeug_log.setLevel(logging.ERROR)
werkzeug_log.disabled = True
app.logger.disabled = True

# -----------------------------
# Feature engineering
# -----------------------------
def create_time_features(data):
    data = data.copy()
    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day
    data["day_of_week"] = data["date"].dt.dayofweek
    data["day_of_year"] = data["date"].dt.dayofyear
    data["week_of_year"] = data["date"].dt.isocalendar().week.astype(int)
    data["quarter"] = data["date"].dt.quarter
    data["is_weekend"] = (data["date"].dt.dayofweek >= 5).astype(int)
    data["is_month_start"] = data["date"].dt.is_month_start.astype(int)
    data["is_month_end"] = data["date"].dt.is_month_end.astype(int)

    data["lag_1"] = data["sales"].shift(1)
    data["lag_7"] = data["sales"].shift(7)
    data["lag_14"] = data["sales"].shift(14)
    data["lag_30"] = data["sales"].shift(30)

    data["rolling_mean_3"] = data["sales"].shift(1).rolling(3).mean()
    data["rolling_mean_7"] = data["sales"].shift(1).rolling(7).mean()
    data["rolling_mean_14"] = data["sales"].shift(1).rolling(14).mean()
    data["rolling_mean_30"] = data["sales"].shift(1).rolling(30).mean()
    return data

FEATURE_COLUMNS = [
    "year", "month", "day", "day_of_week", "day_of_year",
    "week_of_year", "quarter", "is_weekend", "is_month_start", "is_month_end",
    "lag_1", "lag_7", "lag_14", "lag_30",
    "rolling_mean_3", "rolling_mean_7", "rolling_mean_14", "rolling_mean_30"
]

# -----------------------------
# Plot helper
# -----------------------------
def fig_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    buffer.seek(0)
    encoded = base64.b64encode(buffer.read()).decode("utf-8")
    plt.close(fig)
    return encoded

# -----------------------------
# Forecast logic
# -----------------------------
def run_forecast(specific_brand, forecast_days, return_plot_images=False):
    specific_brand = str(specific_brand).strip()
    forecast_days = int(forecast_days)

    if not specific_brand:
        raise ValueError("Brand/productName cannot be empty.")

    if not 1 <= forecast_days <= 90:
        raise ValueError("Forecast days must be between 1 and 90.")

    raw_df = pd.read_csv(file_path)
    raw_df.columns = raw_df.columns.str.strip()

    required_cols = {"saleDate", "totalAmount", "productName"}
    missing_cols = required_cols - set(raw_df.columns)
    if missing_cols:
        raise ValueError(f"Missing columns in CSV: {missing_cols}")

    df = raw_df[["saleDate", "totalAmount", "productName"]].copy()
    df.columns = ["date", "sales", "product_name"]

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
    df["product_name"] = df["product_name"].astype(str).str.strip()

    df = df.dropna(subset=["date", "product_name"])
    df["sales"] = df["sales"].interpolate(method="linear").bfill().ffill()
    df["date"] = df["date"].dt.floor("D")

    daily_product_df = df.groupby(["date", "product_name"], as_index=False)["sales"].sum()
    daily_total_df = daily_product_df.groupby("date", as_index=False)["sales"].sum()

    full_dates = pd.DataFrame({
        "date": pd.date_range(daily_total_df["date"].min(), daily_total_df["date"].max(), freq="D")
    })

    daily_total_df = full_dates.merge(daily_total_df, on="date", how="left")
    daily_total_df["sales"] = daily_total_df["sales"].fillna(0)
    daily_total_df = daily_total_df.sort_values("date").reset_index(drop=True)

    brand_daily = (
        daily_product_df[daily_product_df["product_name"] == specific_brand]
        .groupby("date", as_index=False)["sales"].sum()
        .rename(columns={"sales": "brand_sales"})
    )

    if brand_daily.empty:
        products = sorted(df["product_name"].dropna().unique())
        raise ValueError(f"Brand '{specific_brand}' not found. Sample products: {products[:10]}")

    brand_daily = full_dates.merge(brand_daily, on="date", how="left")
    brand_daily["brand_sales"] = brand_daily["brand_sales"].fillna(0)
    brand_daily = brand_daily.sort_values("date").reset_index(drop=True)

    top_products = (
        daily_product_df.groupby("product_name")["sales"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .index
    )
    plot_product_df = daily_product_df[daily_product_df["product_name"].isin(top_products)].copy()

    brand_model_input = brand_daily.rename(columns={"brand_sales": "sales"})[["date", "sales"]]
    brand_feature_df = create_time_features(brand_model_input)
    brand_model_df = brand_feature_df.dropna().reset_index(drop=True)

    if len(brand_model_df) < 40:
        raise ValueError("Not enough brand history to forecast. Try a brand with more daily data.")

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=3,
        min_samples_leaf=2,
        random_state=42
    )
    model.fit(brand_model_df[FEATURE_COLUMNS], brand_model_df["sales"])

    forecast_base = brand_model_input.copy()
    future_rows = []

    for _ in range(forecast_days):
        next_date = forecast_base["date"].max() + pd.Timedelta(days=1)
        new_row = pd.DataFrame({"date": [next_date], "sales": [None]})
        forecast_base = pd.concat([forecast_base, new_row], ignore_index=True)

        temp_df = create_time_features(forecast_base)
        next_features = temp_df.iloc[[-1]][FEATURE_COLUMNS]
        next_prediction = float(model.predict(next_features)[0])

        forecast_base.loc[forecast_base.index[-1], "sales"] = next_prediction
        future_rows.append({
            "date": next_date.strftime("%Y-%m-%d"),
            "predicted_sales": round(next_prediction, 2)
        })

    future_forecast = pd.DataFrame(future_rows)

    output_file = f"{specific_brand}_forecast_{forecast_days}d.csv".replace(" ", "_")
    output_path = os.path.abspath(output_file)
    future_forecast.to_csv(output_path, index=False)

    plot_images = []

    if return_plot_images:
        fig1 = plt.figure(figsize=(16, 7))
        sns.lineplot(data=plot_product_df, x="date", y="sales", hue="product_name", linewidth=2)
        plt.title("Historical Daily Sales by Product Name")
        plt.xlabel("Date")
        plt.ylabel("Sales")
        plt.legend(title="Product Name", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plot_images.append(fig_to_base64(fig1))

        fig2 = plt.figure(figsize=(16, 6))
        plt.plot(daily_total_df["date"], daily_total_df["sales"], color="navy", linewidth=2, label="Total Historical Daily Sales")
        plt.title("Total Historical Daily Sales")
        plt.xlabel("Date")
        plt.ylabel("Sales")
        plt.legend()
        plt.tight_layout()
        plot_images.append(fig_to_base64(fig2))

        fig3 = plt.figure(figsize=(16, 6))
        plt.plot(brand_daily["date"], brand_daily["brand_sales"], label=f"{specific_brand} Historical Sales", linewidth=2, color="teal")
        plt.plot(pd.to_datetime(future_forecast["date"]), future_forecast["predicted_sales"], label=f"{specific_brand} Forecast Sales", linewidth=2, color="darkorange")
        plt.title(f"{specific_brand} Forecast for Next {forecast_days} Days")
        plt.xlabel("Date")
        plt.ylabel("Sales")
        plt.legend()
        plt.tight_layout()
        plot_images.append(fig_to_base64(fig3))

    return future_forecast, output_path, plot_images

# -----------------------------
# HTML
# -----------------------------
HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Sales Forecast App</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 1100px; margin: 40px auto; padding: 20px; background: #f8fafc; }
        .card { background: white; padding: 24px; border-radius: 14px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); }
        h1, h2 { color: #0f172a; }
        p { color: #475569; }
        label { display: block; margin-top: 16px; margin-bottom: 6px; font-weight: bold; color: #0f172a; }
        input { width: 100%; padding: 10px; border: 1px solid #cbd5e1; border-radius: 8px; font-size: 14px; box-sizing: border-box; }
        button { margin-top: 18px; background: #0f766e; color: white; border: none; padding: 12px 18px; border-radius: 8px; cursor: pointer; }
        .error { color: #b91c1c; margin-top: 16px; font-weight: bold; }
        .success { color: #166534; margin-top: 16px; font-weight: bold; }
        table { width: 100%; margin-top: 20px; border-collapse: collapse; background: white; }
        th, td { border: 1px solid #e2e8f0; padding: 10px; text-align: left; }
        th { background: #e2e8f0; }
        code { background: #f1f5f9; padding: 2px 6px; border-radius: 6px; }
        .hint { margin-top: 14px; font-size: 14px; color: #475569; }
        .plot-block { margin-top: 28px; }
        .plot-block img { width: 100%; border: 1px solid #e2e8f0; border-radius: 10px; background: white; }
    </style>
</head>
<body>
    <div class="card">
        <h1>Sales Forecast App</h1>
        <p>Enter a product name and forecast days to generate future sales predictions.</p>

        <form method="post" action="/forecast-form">
            <label for="product_name">Product Name</label>
            <input type="text" id="product_name" name="product_name" value="{{ form_product_name or '' }}" required>

            <label for="forecast_days">Forecast Days (1 to 90)</label>
            <input type="number" id="forecast_days" name="forecast_days" min="1" max="90" value="{{ form_forecast_days or 30 }}" required>

            <button type="submit">Generate Forecast</button>
        </form>

        <div class="hint">JSON API endpoint: <code>POST /forecast</code></div>

        {% if error %}<div class="error">{{ error }}</div>{% endif %}
        {% if success %}<div class="success">{{ success }}</div>{% endif %}

        {% if records %}
        <h2>Forecast Table</h2>
        <table>
            <thead><tr><th>Date</th><th>Predicted Sales</th></tr></thead>
            <tbody>
                {% for row in records %}
                <tr><td>{{ row["date"] }}</td><td>{{ row["predicted_sales"] }}</td></tr>
                {% endfor %}
            </tbody>
        </table>
        <div class="hint">CSV saved at: <code>{{ csv_path }}</code></div>
        {% endif %}

        {% if plot_images %}
        <div class="plot-block"><h2>Graph 1</h2><img src="data:image/png;base64,{{ plot_images[0] }}"></div>
        <div class="plot-block"><h2>Graph 2</h2><img src="data:image/png;base64,{{ plot_images[1] }}"></div>
        <div class="plot-block"><h2>Graph 3</h2><img src="data:image/png;base64,{{ plot_images[2] }}"></div>
        {% endif %}
    </div>
</body>
</html>
"""

# -----------------------------
# Routes
# -----------------------------
@app.route("/", methods=["GET"])
def home():
    return render_template_string(
        HTML_PAGE,
        error=None,
        success=None,
        records=None,
        csv_path=None,
        plot_images=None,
        form_product_name="",
        form_forecast_days=30
    )

@app.route("/forecast-form", methods=["POST"])
def forecast_form():
    specific_brand = request.form.get("product_name", "").strip()
    forecast_days_raw = request.form.get("forecast_days", "30")
    try:
        forecast_days = int(forecast_days_raw)
        future_forecast, output_path, plot_images = run_forecast(
            specific_brand=specific_brand,
            forecast_days=forecast_days,
            return_plot_images=True
        )
        return render_template_string(
            HTML_PAGE,
            error=None,
            success=f"Forecast generated successfully for {specific_brand}",
            records=future_forecast.to_dict(orient="records"),
            csv_path=output_path,
            plot_images=plot_images,
            form_product_name=specific_brand,
            form_forecast_days=forecast_days
        )
    except Exception as e:
        return render_template_string(
            HTML_PAGE,
            error=str(e),
            success=None,
            records=None,
            csv_path=None,
            plot_images=None,
            form_product_name=specific_brand,
            form_forecast_days=forecast_days_raw
        )

@app.route("/forecast", methods=["POST"])
def forecast_api():
    try:
        data = request.get_json(force=True)
        specific_brand = str(data.get("product_name", "")).strip()
        forecast_days = int(data.get("forecast_days", 30))
        future_forecast, output_path, _ = run_forecast(
            specific_brand=specific_brand,
            forecast_days=forecast_days,
            return_plot_images=False
        )
        return jsonify({
            "status": "success",
            "product_name": specific_brand,
            "forecast_days": forecast_days,
            "csv_path": output_path,
            "forecast": future_forecast.to_dict(orient="records")
        })
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

@app.route("/favicon.ico")
def favicon():
    return "", 204

# -----------------------------
# Silent Flask server
# -----------------------------
class SilentWSGIServer:
    def __init__(self, flask_app, host="127.0.0.1", port=5001):
        self.server = make_server(host, port, flask_app)
        self.ctx = flask_app.app_context()
        self.ctx.push()

    def serve_forever(self):
        self.server.serve_forever()

def start_flask():
    server = SilentWSGIServer(app, host="127.0.0.1", port=FLASK_PORT)
    server.serve_forever()

flask_thread = threading.Thread(target=start_flask, daemon=True)
flask_thread.start()

print(f"Flask app is running at http://127.0.0.1:{FLASK_PORT}")
print(f"Browser form: http://127.0.0.1:{FLASK_PORT}")
print(f"API endpoint: POST http://127.0.0.1:{FLASK_PORT}/forecast")

Flask app is running at http://127.0.0.1:5001
Browser form: http://127.0.0.1:5001
API endpoint: POST http://127.0.0.1:5001/forecast
